## The Data Layer

### **Data Scraping**

**Games form top 5 leagues and Ucl**

In [1]:
import pandas as pd
import soccerdata as sd
from Configuration import RAW_DOMESTIC_PATH, RAW_UCL_PATH, PROCESSED_COMBINED_PATH, FIXTURES_TO_PREDICT_PATH

# Define Seasons to Scrape
SEASONS_DOMESTIC = ['2122', '2223', '2324', '2425', '2526', '2627']
SEASONS_UCL = ['2021', '2022', '2023', '2024', '2025', '2026']

# Scrape & Process Domestic Data
print("Scraping Domestic Leagues from FBref...")
fbref = sd.FBref(leagues='Big 5 European Leagues Combined', seasons=SEASONS_DOMESTIC , headless=True)
fbref.rate_limit = 12 # Set rate limit to avoid being blocked by FBref
schedule_domestic = fbref.read_schedule()

cols_domestic = ['league', 'season', 'game_id', 'date', 'home_team', 'away_team', 'home_score', 'away_score', 'home_xg', 'away_xg']
available_cols = [c for c in cols_domestic if c in schedule_domestic.columns]
schedule_domestic = schedule_domestic[available_cols].copy()

# Split domestic into played and unplayed
played_domestic = schedule_domestic.dropna(subset=['home_score', 'away_score']).copy()
unplayed_domestic = schedule_domestic[schedule_domestic['home_score'].isna() & (schedule_domestic['season'] == '2627')].copy()

played_domestic.to_csv(RAW_DOMESTIC_PATH, index=False)

# Scrape & Process UCL Data
print("\nScraping Champions League from MatchHistory...")
ucl_scraper = sd.MatchHistory(leagues=['INT-Champions League'], seasons=SEASONS_UCL)
ucl_raw = ucl_scraper.read_games().reset_index()

season_mapping = {
    '2021': '2122', '2022': '2223', '2023': '2324', 
    '2024': '2425', '2025': '2526', '2026': '2627'
}

ucl_aligned = pd.DataFrame()
ucl_aligned['league'] = ucl_raw['league']
ucl_aligned['season'] = ucl_raw['season'].astype(str).map(season_mapping).fillna(ucl_raw['season'])
ucl_aligned['game_id'] = ucl_raw.get('game_id', ucl_raw.index.map(lambda x: f"ucl_{x}"))
ucl_aligned['date'] = pd.to_datetime(ucl_raw['date'])
ucl_aligned['home_team'] = ucl_raw['home_team']
ucl_aligned['away_team'] = ucl_raw['away_team']
ucl_aligned['home_score'] = ucl_raw['home_score']
ucl_aligned['away_score'] = ucl_raw['away_score']
ucl_aligned['home_xg'] = pd.NA
ucl_aligned['away_xg'] = pd.NA

# Split UCL into played and unplayed
played_ucl = ucl_aligned.dropna(subset=['home_score', 'away_score']).copy()
unplayed_ucl = ucl_aligned[ucl_aligned['home_score'].isna() & (ucl_aligned['season'] == '2627')].copy()

played_ucl.to_csv(RAW_UCL_PATH, index=False)

# Merge Historical Results (Model Training Set)
print("\nCombining historical datasets...")
master_df = pd.concat([played_domestic, played_ucl], ignore_index=True)
master_df.to_csv(PROCESSED_COMBINED_PATH, index=False)

# Merge Upcoming Fixtures (Model Prediction Target Set)
print("Combining upcoming fixtures...")
fixtures_to_predict = pd.concat([unplayed_domestic, unplayed_ucl], ignore_index=True)
fixtures_to_predict.to_csv(FIXTURES_TO_PREDICT_PATH, index=False)

print(f"\nPipeline Complete!")
print(f"-> Historical Training Data: {PROCESSED_COMBINED_PATH} ({len(master_df)} matches)")
print(f"-> Unplayed 2026/27 Fixtures: {FIXTURES_TO_PREDICT_PATH} ({len(fixtures_to_predict)} matches)")


[08/04/26 19:22:49] INFO     No custom team name replacements found. You can configure these in       _config.py:91
                             C:\Users\WAFACo\soccerdata\config\teamname_replacements.json.                         

                    INFO     No custom league dict found. You can configure additional leagues in    _config.py:189
                             C:\Users\WAFACo\soccerdata\config\league_dict.json.                                   

Scraping Domestic Leagues from FBref...


                    INFO     Saving cached data to C:\Users\WAFACo\soccerdata\data\FBref             _common.py:250


*** chromedriver to download = 150.0.7871.124 (Previous Version)

https://storage.googleapis.com/chrome-for-testing-public/150.0.7871.124/win64/chromedriver-win64.zip ...
Download Complete!

Extracting ['chromedriver.exe'] from chromedriver-win64.zip ...
Unzip Complete!

The file [chromedriver.exe] was saved to:
E:\yamen projects\Machine Learning\UCL Prediction Tool\.venv\Lib\site-packages\seleniumbase\drivers\
chromedriver.exe

Making [chromedriver.exe 150.0.7871.124] executable ...
[chromedriver.exe 150.0.7871.124] is now ready for use!



KeyError: ['home_score', 'away_score']